In [ ]:
# ── Setup Cell 1: Mount Google Drive (run once per session) ──────────────────
# Drive is the persistence layer: checkpoints, LMs, and outputs survive
# session restarts here even though /content/ is wiped on disconnect.
#
# Before running: Runtime → Change runtime type → T4 GPU

from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
Path(CHECKPOINTS_DIR).mkdir(parents=True, exist_ok=True)
print(f"Drive mounted. Checkpoints directory: {CHECKPOINTS_DIR}")

In [ ]:
# ── Setup Cell 2: Clone repo and install package ─────────────────────────────
# Clones to /content/repo (ephemeral). Re-run at the start of every session.
# Before running: Runtime → Change runtime type → T4 GPU
#
# Uses --no-deps to avoid Colab pip resolver conflicts: torch/torchvision/
# numpy/scipy are already installed at GPU-compatible versions by Colab and
# should not be upgraded. Setup Cell 3 installs remaining non-Colab dependencies.

import os, subprocess, sys

REPO_URL = "https://github.com/violasarah2000/hackingrongo.git"
REPO_DIR = "/content/repo"

if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "origin"])
    subprocess.check_call(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"])
    print("Repo updated to latest commit.")

os.chdir(REPO_DIR)
# --no-deps: register the package in editable mode without touching Colab's
# pre-installed torch/torchvision/numpy/scipy (Setup Cell 3 handles everything else).
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"
])
print("Package installed. Working directory:", os.getcwd())

In [ ]:
# ── Setup Cell 3: Extra dependencies ───────────────────────────────────────────
# Installs packages not pre-loaded by Colab's base image.
# torch / torchvision / numpy / scipy / Pillow are already present — skip them.
# Packages are installed in separate calls so a single failure is identifiable.
#
# SA solver uses a pure numpy fallback — no D-Wave packages required.

import subprocess, sys

def pip(*args):
    """Run pip install; print output only on failure."""
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *args],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f"FAILED: pip install {' '.join(args)}")
        print(result.stdout[-2000:] if result.stdout else "")
        print(result.stderr[-2000:] if result.stderr else "")
        raise RuntimeError(f"pip install failed for: {args}")
    print(f"  ✓ {', '.join(args)}")

pip("hydra-core>=1.3", "omegaconf>=2.3")
pip("click>=8.1", "jinja2>=3.1")
pip("dimod>=0.12")
pip("cairosvg")
pip("umap-learn", "hdbscan")
pip("gdown")
pip("dwave-samplers>=1.0")  # provides dwave.samplers.SimulatedAnnealingSampler (neal replacement)

print("\nAll extra dependencies installed.")

In [ ]:
# ── Setup Cell 4: Download data and restore checkpoints ───────────────────────
import subprocess, shutil, zipfile, os
from pathlib import Path

REPO_DIR = "/content/repo"
DRIVE_FILE_ID = "1ywwTdqvFvrTV6i6zjBJkiWLT4GjmKDn4"
ZIP_PATH = Path("/content/hackingrongo_data.zip")
DATA_DIR = Path(REPO_DIR) / "data"
FORCE_REEXTRACT = False

if not ZIP_PATH.exists():
    print("Downloading zip from Google Drive...")
    subprocess.check_call(["gdown", DRIVE_FILE_ID, "-O", str(ZIP_PATH)])
    print("Download complete.")
else:
    print("Zip already present — skipping download.")

_sentinel = DATA_DIR / "corpus" / "A.json"
if _sentinel.exists() and not FORCE_REEXTRACT:
    n_corpus = len(list((DATA_DIR / "corpus").glob("*.json")))
    print(f"Corpus already extracted ({n_corpus} JSON files) — skipping.")
    print("  Set FORCE_REEXTRACT = True to overwrite.")
else:
    TMPDIR = Path("/content/hackingrongo_extract")
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR.mkdir()
    print(f"Extracting data/ from {ZIP_PATH} ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        data_members = [m for m in zf.namelist() if m.startswith("data/")]
        if not data_members:
            raise RuntimeError(
                "Zip contains no data/ folder. Check the zip was created from "
                "inside the hackingrongo/ project root."
            )
        zf.extractall(TMPDIR, members=data_members)
    extracted_data = TMPDIR / "data"
    DATA_DIR.mkdir(exist_ok=True)
    for item in extracted_data.iterdir():
        dest = DATA_DIR / item.name
        if dest.exists():
            shutil.rmtree(dest) if dest.is_dir() else dest.unlink()
        shutil.move(str(item), str(dest))
    shutil.rmtree(TMPDIR)
    n_corpus = len(list((DATA_DIR / "corpus").glob("*.json")))
    svgs = list(DATA_DIR.rglob("*.svg"))
    pngs = list(DATA_DIR.rglob("*.png"))
    print(f"Done: {n_corpus} corpus JSONs, {len(svgs)} SVGs, {len(pngs)} PNGs in {DATA_DIR}")

# ── Restore checkpoints from Drive ────────────────────────────────────────────
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
ckpt_path = Path(CHECKPOINTS_DIR)
if ckpt_path.exists():
    restored = 0
    for f in ckpt_path.glob("*.pt"):
        dest = Path(REPO_DIR) / "outputs" / f.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(f, dest)
        print(f"  Restored: {f.name}  ({f.stat().st_size/1024/1024:.1f} MB)")
        restored += 1
    lm_dest = DATA_DIR / "language_models"
    lm_dest.mkdir(parents=True, exist_ok=True)
    for f in ckpt_path.glob("*_lm.json"):
        shutil.copy(f, lm_dest / f.name)
        print(f"  Restored LM: {f.name}")
        restored += 1
    pv = ckpt_path / "parallel_variants_auto.json"
    if pv.exists():
        par_dest = DATA_DIR / "parallels"
        par_dest.mkdir(parents=True, exist_ok=True)
        shutil.copy(pv, par_dest / pv.name)
        print("  Restored: parallel_variants_auto.json")
        restored += 1
    print(f"\nRestored {restored} file(s) from Drive.")
else:
    print("No Drive checkpoints found — fresh start.")

print("\nSetup Cell 5 (smoke test) is ready to run.")

# ── 3D tablet crops (optional, pre-processed asset) ──────────────────────────
# The NXZ mesh files (data/glyphs/3d_models/) cannot be converted in Colab.
# To use 3D-derived glyph crops in Zone A training, pre-process them locally:
#
#   pip install playwright && playwright install chromium
#   python scripts/render_tablet_views.py --tablet all --num-views 24 \
#       --width 2048 --height 2048 --output-dir data/glyphs/synthetic_views/
#   for T in B C D; do
#     for S in r v; do
#       python scripts/segment_3d_glyphs.py --tablet $T --side $S \
#           --renders data/glyphs/synthetic_views/tablet_${T}/ \
#           --corpus  data/corpus/${T}.json \
#           --output  data/glyphs/3d_crops/ --crop-size 64
#     done
#   done
#
# Then add data/glyphs/3d_crops/ to the Drive data zip and re-upload.
# GlyphImageDataset will automatically use the crops as a fallback source
# for any Barthel code not already covered by barthel_ref/ or barthel_corpus/.
#
# If 3d_crops/ is present in the data zip it is extracted automatically above.

In [ ]:
# ── Setup Cell 5: Smoke test — verify imports and data ───────────────────────
# Runs quickly (< 10 seconds). Must pass before running Zone A Cell 1.

import os, json
from pathlib import Path

REPO_DIR = "/content/repo"
DATA_DIR = f"{REPO_DIR}/data"
os.chdir(REPO_DIR)

print("── Import check ──────────────────────────────────────────")
import hackingrongo
from hackingrongo.data.corpus import load_corpus
from hackingrongo.zone_a.autoencoder import ConvAutoencoder
from hackingrongo.zone_b.entropy import index_of_coincidence
from hackingrongo.zone_c.mcmc import MCMCSampler
print("  ✓ All core imports OK")

print("\n── Corpus check ──────────────────────────────────────────")
corpus_dir = Path(f"{DATA_DIR}/corpus")
tablet_files = sorted(corpus_dir.glob("*.json"))
print(f"  Tablet files: {len(tablet_files)}")
assert len(tablet_files) > 0, "No tablet JSON files found — run Setup Cell 4 first"

total_tokens = 0
for tf in tablet_files[:3]:
    d = json.loads(tf.read_text())
    n = len(d.get("glyphs", []))
    total_tokens += n
    print(f"  {tf.stem}: {n} tokens")
print(f"  ✓ Corpus readable")

print("\n── GPU check ─────────────────────────────────────────────")
import torch
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
else:
    print("  ⚠ No GPU — Zone A training will be slow. Runtime → Change runtime type → T4 GPU")

print("\n✓ Smoke test passed. Zone A Cell 1 (training) is ready.")

In [ ]:
# ── Zone A Cell 1: Train autoencoder ─────────────────────────────────────────
# Auto-resumes from the latest checkpoint on Drive.
# If 50 epochs are already complete, skips training and re-extracts embeddings.
# To force a full retrain from scratch, set resume_checkpoint to "none" below
# or delete autoencoder_epoch*.pt from the Drive checkpoints directory.
#
# Outputs saved to Drive:
#   autoencoder_epoch*.pt          ← training checkpoints
#   embeddings_cache.pt            ← glyph embedding vectors (input to Zone A Cell 2)

import subprocess, sys, os, shutil
from pathlib import Path
import torch

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"
EMBEDDINGS_CACHE = Path(f"{REPO_DIR}/outputs/embeddings_cache.pt")

os.chdir(REPO_DIR)

ckpt_dst = Path(REPO_DIR) / "outputs" / "checkpoints"
ckpt_dst.mkdir(parents=True, exist_ok=True)

# ── Restore checkpoints from Drive ────────────────────────────────────────────
# Skip any that don't match the current model architecture.
from hackingrongo.zone_a.autoencoder import ConvAutoencoder
import omegaconf
_cfg = omegaconf.OmegaConf.load(f"{REPO_DIR}/conf/config.yaml")
_model = ConvAutoencoder(_cfg)
_current_keys = set(_model.state_dict().keys())
del _model, _cfg

for f in sorted(Path(CHECKPOINTS_DIR).glob("autoencoder_*.pt")):
    try:
        ckpt = torch.load(f, map_location="cpu", weights_only=True)
        ckpt_keys = set(ckpt["model_state_dict"].keys())
        if ckpt_keys != _current_keys:
            raise RuntimeError("key mismatch: old arch")
        shutil.copy(f, ckpt_dst / f.name)
        print(f"  Restored checkpoint: {f.name}")
    except Exception as e:
        print(f"  Skipped incompatible checkpoint {f.name}: {e}")

# ── Pre-flight: validate or purge existing embeddings cache ───────────────────
print("=" * 60)
print("PRE-FLIGHT: Checking embeddings cache")
print("=" * 60)

CACHE_VALID = False
if EMBEDDINGS_CACHE.exists():
    try:
        _d = torch.load(EMBEDDINGS_CACHE, map_location="cpu", weights_only=True)
        _has_embeddings = isinstance(_d, dict) and "embeddings" in _d
        _has_codes      = _has_embeddings and any(_d.get("barthel_codes", []))
        if _has_embeddings and _has_codes:
            _n       = len(_d["embeddings"])
            _n_coded = sum(1 for c in _d["barthel_codes"] if c)
            print(f"  Cache OK: {_n} embeddings, {_n_coded} with Barthel codes.")
            print("  Skipping training — delete embeddings_cache.pt on Drive to retrain.")
            CACHE_VALID = True
        else:
            reason = []
            if not _has_embeddings: reason.append("missing 'embeddings' key")
            if not _has_codes:      reason.append("barthel_codes empty")
            print(f"  Cache invalid ({'; '.join(reason)}) — deleting and retraining.")
            EMBEDDINGS_CACHE.unlink()
        del _d
    except Exception as e:
        print(f"  Cache unreadable ({e}) — deleting and retraining.")
        EMBEDDINGS_CACHE.unlink()
else:
    print("  No cache found — will train (auto-resumes from latest checkpoint if any).")

# ── Training ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 1: Training Zone A autoencoder")
print("=" * 60)

if CACHE_VALID:
    print("  Skipped (valid cache already present).")
else:
    proc = subprocess.Popen(
        [
            sys.executable, "scripts/train_autoencoder.py",
            f"paths.corpus_dir={DATA_DIR}/corpus",
            f"paths.glyphs_dir={DATA_DIR}/glyphs",
            f"paths.catalog_dir={DATA_DIR}/catalog",
            f"paths.outputs_dir={REPO_DIR}/outputs",
            f"paths.checkpoints_dir={str(ckpt_dst)}",
            f"paths.embeddings_cache={EMBEDDINGS_CACHE}",
            "zone_a.autoencoder.num_epochs=50",
            "zone_a.autoencoder.batch_size=64",
            # "auto" globs checkpoints_dir for the latest .pt and resumes.
            # Change to "none" to force a full retrain from scratch.
            "zone_a.autoencoder.resume_checkpoint=auto",
            "hydra.job.chdir=false",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Zone A training failed (exit {proc.returncode})")
    print(f"\nDone. Embeddings cache at: {EMBEDDINGS_CACHE}")

# ── Verify + save to Drive ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("VERIFICATION + SAVING TO DRIVE")
print("=" * 60)

ckpts = sorted(ckpt_dst.glob("autoencoder_epoch*.pt"))
for f in ckpts:
    dest = Path(CHECKPOINTS_DIR) / f.name
    shutil.copy(f, dest)
    print(f"  ✓ {f.name}  {f.stat().st_size/1024/1024:.1f} MB → Drive")
if not ckpts:
    print("  ✗ No checkpoints found")

if EMBEDDINGS_CACHE.exists():
    shutil.copy(EMBEDDINGS_CACHE, Path(CHECKPOINTS_DIR) / "embeddings_cache.pt")
    print(f"  ✓ embeddings_cache.pt  {EMBEDDINGS_CACHE.stat().st_size/1024/1024:.1f} MB → Drive")
    print("\nZone A Cell 2 is ready to run.")
else:
    print("  ✗ embeddings_cache.pt NOT FOUND — check logs above")

In [ ]:
# ── Zone A Cell 2: Analyse embeddings — UMAP + HDBSCAN + reports ─────────────
# Produces: umap_embeddings.png, cluster_vs_barthel.csv/.json,
#           divergence_report.html, compound_candidates.json,
#           compound_report.html, stratum_glyph_report.html
# Only run after Zone A Cell 1 completes successfully.

import subprocess, sys, os, json
from pathlib import Path
from IPython.display import Image, display

REPO_DIR         = "/content/repo"
CHECKPOINTS_DIR  = "/content/drive/MyDrive/hackingrongo_checkpoints"
EMBEDDINGS_CACHE = f"{REPO_DIR}/outputs/embeddings_cache.pt"
DATA_DIR         = f"{REPO_DIR}/data"
ANALYSIS_OUT     = f"{REPO_DIR}/outputs/analysis"

os.chdir(REPO_DIR)
Path(ANALYSIS_OUT).mkdir(parents=True, exist_ok=True)

if not Path(EMBEDDINGS_CACHE).exists():
    raise FileNotFoundError("embeddings_cache.pt not found. Run Zone A Cell 1 first.")
print(f"Cache: {Path(EMBEDDINGS_CACHE).stat().st_size/1024/1024:.1f} MB  ✓")

# ── Step 1: UMAP + HDBSCAN + divergence report ────────────────────────────────
print("\n" + "=" * 60)
print("STEP 1: Embedding analysis + divergence report")
print("=" * 60)

proc = subprocess.Popen(
    [
        sys.executable, "scripts/analyze_embeddings.py",
        f"paths.embeddings_cache={EMBEDDINGS_CACHE}",
        f"paths.outputs_dir={REPO_DIR}/outputs",
        "hydra.job.chdir=false",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Analysis failed (exit {proc.returncode})")

# ── Step 2: Compound detector — validation run ────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: Compound detector — P@k validation on known compounds")
print("=" * 60)

proc = subprocess.Popen(
    [
        sys.executable, "-m", "hackingrongo.zone_b.compound_detector",
        "--analysis-dir", "outputs/analysis",
        "--corpus-dir", f"{DATA_DIR}/corpus",
        "--output", "outputs/analysis/compound_validation.json",
        "--include-known", "--min-methods", "2", "--min-confidence", "0.25",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: Validation run failed (exit {proc.returncode}) — continuing")

val_path = Path(ANALYSIS_OUT) / "compound_validation.json"
if val_path.exists():
    val   = json.loads(val_path.read_text())
    cands = val.get("candidates", [])
    print("\nPrecision@k on known compounds:")
    for k in [5, 10, 20]:
        top     = cands[:k]
        n_known = sum(1 for c in top if c.get("is_known_compound", False))
        p       = n_known / max(len(top), 1)
        bar     = "█" * round(p * 20) + "░" * (20 - round(p * 20))
        print(f"  P@{k:2d}: {p:.2f}  {bar}  ({n_known}/{len(top)})")

# ── Step 3: Compound detector — novel candidates ──────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: Compound detector — novel candidates")
print("=" * 60)

proc = subprocess.Popen(
    [
        sys.executable, "-m", "hackingrongo.zone_b.compound_detector",
        "--analysis-dir", "outputs/analysis",
        "--corpus-dir", f"{DATA_DIR}/corpus",
        "--output", "outputs/analysis/compound_candidates.json",
        "--min-methods", "2", "--min-confidence", "0.25",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: Detection failed (exit {proc.returncode}) — continuing")

# ── Step 4: Compound report ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: Compound report")
print("=" * 60)

if (Path(ANALYSIS_OUT) / "compound_candidates.json").exists():
    proc = subprocess.Popen(
        [
            sys.executable, "-m", "hackingrongo.results.compound_report",
            "--candidates", "outputs/analysis/compound_candidates.json",
            "--svg-catalog", f"{DATA_DIR}/glyphs/svg/catalog.json",
            "--output", "outputs/analysis/compound_report.html",
            "--max-candidates", "50",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
else:
    print("No compound_candidates.json — skipping report")

# ── Step 5: Parallel passage cross-reference ──────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: Parallel passage cross-reference")
print("=" * 60)

proc = subprocess.Popen(
    [
        sys.executable, "scripts/cross_reference_parallels.py",
        "--input",     f"{DATA_DIR}/parallels/horley_parallels.csv",
        "--corpus",    f"{DATA_DIR}/corpus",
        "--config",    f"{REPO_DIR}/conf/config.yaml",
        "--tablets",   f"{DATA_DIR}/metadata/tablets.json",
        "--output",    f"{DATA_DIR}/parallels/parallel_variants_auto.json",
        "--threshold", "1",
        "--verbose",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: Cross-reference failed (exit {proc.returncode}) — continuing")

# ── Step 6: Diachronic passage report ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: Diachronic passage report")
print("=" * 60)

variants_path = Path(f"{DATA_DIR}/parallels/parallel_variants_auto.json")
if variants_path.exists():
    proc = subprocess.Popen(
        [
            sys.executable, "-m", "hackingrongo.results.passage_report",
            "--input",        str(variants_path),
            "--output",       f"{REPO_DIR}/outputs/analysis/passage_reports",
            "--filter-score", "0.0",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        print(f"WARNING: Passage report failed (exit {proc.returncode}) — continuing")
else:
    print("parallel_variants_auto.json not found — skipping passage report")

# ── Step 7: Display cluster summary + UMAP ───────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: Results")
print("=" * 60)

json_path = Path(ANALYSIS_OUT) / "cluster_vs_barthel.json"
if json_path.exists():
    result = json.loads(json_path.read_text())
    print(f"  Embeddings : {result['n_embeddings']}")
    print(f"  Clusters   : {result['n_clusters']}")
    print(f"  Noise pts  : {result['n_noise_points']}")
    ari = result.get("adjusted_rand_index")
    if ari is not None:
        print(f"  ARI        : {ari:.4f}  ({result.get('interpretation', '')})")
    print("\nTop 5 clusters by size:")
    clusters = {k: v for k, v in result.get("clusters", {}).items() if k != "noise"}
    for cid, info in sorted(clusters.items(), key=lambda x: -x[1]["size"])[:5]:
        top_codes = list(info["barthel_codes"].keys())[:4]
        top_fams  = list(info["barthel_families"].keys())[:2]
        print(f"  Cluster {cid}: {info['size']} glyphs  codes={top_codes}  families={top_fams}")

plot_path = Path(ANALYSIS_OUT) / "umap_embeddings.png"
if plot_path.exists():
    print("\nUMAP scatter:")
    display(Image(filename=str(plot_path)))

# ── Step 8: Astronomical glyph candidate analysis ────────────────────────────
print("\n" + "=" * 60)
print("STEP 8: Astronomical glyph candidate analysis")
print("=" * 60)

Path(f"{REPO_DIR}/outputs/zone_b").mkdir(parents=True, exist_ok=True)
proc = subprocess.Popen(
    [
        sys.executable, "-m", "hackingrongo.zone_b.astronomical_analysis",
        "--corpus-dir", f"{DATA_DIR}/corpus",
        "--output",     f"{REPO_DIR}/outputs/zone_b/astronomical_candidates.json",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: Astronomical analysis failed (exit {proc.returncode}) — continuing")

astro_path = Path(f"{REPO_DIR}/outputs/zone_b/astronomical_candidates.json")
if astro_path.exists():
    try:
        astro_data  = json.loads(astro_path.read_text())
        candidates  = astro_data.get("candidates", [])
        print(f"\n  Candidates flagged: {len(candidates)}")
        if candidates:
            print(f"  {'Code':<8} {'Score':>6}  {'Methods':>7}  Details")
            print(f"  {'-'*8} {'-'*6}  {'-'*7}  {'-'*35}")
            for c in candidates[:10]:
                code   = c.get("barthel_code", "?")
                score  = c.get("overall_score", c.get("consensus_confidence", 0))
                n      = c.get("n_methods_flagged", c.get("n_methods_agreeing", 0))
                entry  = c.get("dietrich_entry")
                if isinstance(entry, dict):
                    desc = entry.get("proposed_referent", "—")
                elif entry is not None:
                    desc = str(entry)
                else:
                    desc = "bird-headed / no Dietrich entry"
                print(f"  {code:<8} {score:>6.3f}  {n:>7}  {desc}")
    except Exception as e:
        print(f"  WARNING: Could not display candidates inline: {e}")

# ── Step 9: Astronomical HTML report ─────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 9: Astronomical report")
print("=" * 60)

if astro_path.exists():
    proc = subprocess.Popen(
        [
            sys.executable, "-m", "hackingrongo.results.astronomical_report",
            "--candidates",  str(astro_path),
            "--svg-catalog", f"{DATA_DIR}/glyphs/svg/catalog.json",
            "--output",      f"{REPO_DIR}/outputs/analysis/astronomical_report.html",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        print(f"WARNING: Astronomical report failed — continuing")
else:
    print("astronomical_candidates.json not found — skipping report")

# ── Step 10: Train sequence model (for glyph reconstruction) ─────────────────
print("\n" + "=" * 60)
print("STEP 10: Train sequence model")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/train_sequence_model.py",
        "--output", f"{REPO_DIR}/outputs/zone_b/sequence_model.json",
        "--order",  "3",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: Sequence model training failed — continuing")
else:
    proc2 = subprocess.Popen(
        [
            sys.executable, "scripts/complete_sequence.py",
            "--tablet", "F",
            "--model",  f"{REPO_DIR}/outputs/zone_b/sequence_model.json",
            "--top-k",  "5",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc2.stdout:
        print(line, end="")
    proc2.wait()

# ── Step 11: Stratum glyph report ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 11: Stratum glyph report (pre-contact / post-contact / both)")
print("=" * 60)

proc = subprocess.Popen(
    [
        sys.executable, "-m", "hackingrongo.results.stratum_glyph_report",
        "--catalog",    f"{DATA_DIR}/glyphs/svg/catalog.json",
        "--tablets",    f"{DATA_DIR}/metadata/tablets.json",
        "--sign-meta",  f"{DATA_DIR}/catalog/sign_metadata.json",
        "--glyphs-dir", f"{DATA_DIR}/glyphs",
        "--output",     f"{REPO_DIR}/outputs/analysis/stratum_glyph_report.html",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: Stratum glyph report failed (exit {proc.returncode}) — continuing")
else:
    sgr = Path(f"{REPO_DIR}/outputs/analysis/stratum_glyph_report.html")
    if sgr.exists():
        print(f"  ✓ stratum_glyph_report.html  ({sgr.stat().st_size/1024:.0f} KB)")

# ── Step 8b: Quantum p_good measurement ───────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8b: Quantum hardness analysis (p_good measurement)")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/measure_pgood.py",
        "--corpus-dir",      f"{DATA_DIR}/corpus",
        "--lm-dir",          f"{REPO_DIR}/data/language_models",
        "--n-samples",       "10000",
        "--thresholds",      "0.90,0.95,0.99",
        "--mcmc-iterations", "5000",
        "--iqae-epsilon", "0.05",
        "--iqae",
        "--output",          f"{REPO_DIR}/outputs/zone_b/pgood_analysis.json",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: p_good measurement failed — continuing")

pgood_path = Path(f"{REPO_DIR}/outputs/zone_b/pgood_analysis.json")
if pgood_path.exists():
    pg = json.loads(pgood_path.read_text())
    print(f"\n  ── Quantum hardness summary ──────────────────────────────")
    print(f"  Score distribution: mean={pg['score_distribution']['mean']:.1f} "
          f"std={pg['score_distribution']['std']:.1f}")
    for t in pg['thresholds']:
        print(f"  τ={t['tau']:.2f}: p_good={t['p_good']:.2e}  "
              f"Grover={t['grover_oracle_calls']:,} calls  "
              f"Classical={t['classical_random_calls']:,}  "
              f"Speedup={t['quantum_speedup_ratio']:.1f}x")

print("\nZone B Cell 2 (LMs + parallel passages) is ready to run.")

In [ ]:
# ── Cell 6d: Zone C fusion layer training (step 4k) ─────────────────────
# Trains the FusionLayer that combines Zone A embeddings + Zone B priors.
# Requires: Zone A Cell 1 (embeddings_cache.pt) and Zone A Cell 2 (compound_candidates.json).
# Output saved to Drive: zone_c/fusion_checkpoint.pt
#
# Safe to skip — Zone C Cell 2 (MCMC) falls back to raw embeddings if the
# checkpoint is absent.

import subprocess, sys, os, shutil
from pathlib import Path

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"
os.chdir(REPO_DIR)

EMBEDDINGS = Path(f"{REPO_DIR}/outputs/embeddings_cache.pt")
COMPOUNDS  = Path(f"{REPO_DIR}/outputs/analysis/compound_candidates.json")
FUSION_OUT = Path(f"{REPO_DIR}/outputs/zone_c/fusion_checkpoint.pt")

if not EMBEDDINGS.exists():
    print("embeddings_cache.pt not found — run Zone A Cell 1 first.")
elif not COMPOUNDS.exists():
    print("compound_candidates.json not found — run Zone A Cell 2 first.")
else:
    FUSION_OUT.parent.mkdir(parents=True, exist_ok=True)
    proc = subprocess.Popen(
        [
            sys.executable, "scripts/train_fusion.py",
            "--embeddings", str(EMBEDDINGS),
            "--compounds",  str(COMPOUNDS),
            "--corpus-dir", f"{DATA_DIR}/corpus",
            "--output",     str(FUSION_OUT),
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        print(f"WARNING: train_fusion failed (exit {proc.returncode}) — continuing")
    elif FUSION_OUT.exists():
        shutil.copy(FUSION_OUT, Path(CHECKPOINTS_DIR) / "fusion_checkpoint.pt")
        print(f"  ✓ fusion_checkpoint.pt ({FUSION_OUT.stat().st_size/1024:.0f} KB) saved to Drive")

In [ ]:
# ── Zone B Cell 1: Reading order tests (basic) ─────────────────────────────────
# Tests 1 & 2 use the sequence model from Zone A Cell 2 (Step 10).
# Tests 3 & 4 (line-boundary entropy + recto/verso ordering) run corpus-only.
# Test 4 resolves Pozdniakov's 1958 open question on reading-side ordering.
#
# Output saved to Drive: reading_order_results.json

import subprocess, sys, os, shutil
from pathlib import Path

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
os.chdir(REPO_DIR)

RESULTS_JSON = Path(f"{REPO_DIR}/outputs/reading_order_results.json")
CORPUS_DIR   = f"{REPO_DIR}/data/corpus"
MODEL_PATH   = f"{REPO_DIR}/outputs/zone_b/sequence_model.json"

proc = subprocess.Popen(
    [sys.executable, "scripts/reading_order_tests.py",
     "--corpus", CORPUS_DIR,
     "--model",  MODEL_PATH,
     "--tests",  "1", "2", "3", "4",
     "--output", str(RESULTS_JSON)],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: reading_order_tests failed (exit {proc.returncode})")

# ── Save to Drive ─────────────────────────────────────────────────────────────
if RESULTS_JSON.exists():
    shutil.copy(RESULTS_JSON, f"{CHECKPOINTS_DIR}/reading_order_results.json")
    print(f"  ✓ reading_order_results.json saved to Drive")

In [ ]:
# ── Zone B Cell 2: Build language models and parallel passages ─────────────────
# Run once after Zone A Cell 2. Takes ~5 minutes.
# Language models are required for Zone C Cell 2.
# Parallel passage cross-reference finds multi-tablet attestations.
#
# Outputs saved to Drive:
#   parallel_variants_auto.json   ← 13 multi-tablet passages found
#   pre_contact_lm.json           ← Thomson 1891 + Roussel 1908 (~1345 forms)
#   post_contact_lm.json          ← Barthel/Blixen/Fischer + IDS (~2754 forms)
#   smoothing_lm.json             ← Hawaiian Corpus Project (~56K types)

import subprocess, sys, json, shutil, os
from pathlib import Path

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"

os.chdir(REPO_DIR)

# ── Step 1: Fetch IDS Rapa Nui vocabulary ─────────────────────────────────────
print("=" * 60)
print("STEP 1: Fetch IDS Rapa Nui vocabulary (Thomson + Roussel)")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/parse_ids.py",
        "--stratify",
        "--cache-dir", f"{DATA_DIR}/cache",
        "--data-dir",  f"{DATA_DIR}/polynesian_texts",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: parse_ids.py failed (exit {proc.returncode}) — continuing")

# ── Step 2: Fetch ABVD cognate neighbours ─────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: Fetch ABVD East Polynesian cognate neighbours")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/fetch_abvd_corpus.py",
        "--with-cognates",
        "--cache-dir", f"{DATA_DIR}/cache",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: fetch_abvd_corpus.py failed — continuing")

# ── Step 2b: Fetch Hawaiian smoothing corpus ──────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2b: Fetch Hawaiian smoothing corpus")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/fetch_hawaiian_corpus.py",
        "--cache-dir", f"{DATA_DIR}/cache",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: fetch_hawaiian_corpus.py failed — continuing")

# ── Step 3: Build language models ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: Build stratified language models")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/build_language_models.py",
        "hydra.job.chdir=false",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"build_language_models.py failed (exit {proc.returncode})")

print("\n── Language model verification ──────────────────────────────")
lm_dir = Path(f"{REPO_DIR}/data/language_models")
lm_ok = True
for lm_name in ["pre_contact_lm", "post_contact_lm", "smoothing_lm"]:
    p = lm_dir / f"{lm_name}.json"
    if p.exists():
        vocab = len(json.loads(p.read_text()).get("vocab", []))
        status = "✓" if vocab >= 50 else "✗ TOO SPARSE"
        print(f"  {status} {lm_name}: vocab={vocab} types")
        if vocab < 50:
            lm_ok = False
    else:
        print(f"  ✗ {lm_name}.json MISSING")
        lm_ok = False

if not lm_ok:
    print("\nWARNING: LMs too sparse — Zone C will produce uninformative results")
else:
    print("\nLMs ready for Zone C.")

# ── Step 4: Parallel passage cross-reference ──────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: Parallel passage cross-reference")
print("=" * 60)
proc = subprocess.Popen(
    [
        sys.executable, "scripts/cross_reference_parallels.py",
        "--input",     f"{DATA_DIR}/parallels/horley_parallels.csv",
        "--corpus",    f"{DATA_DIR}/corpus",
        "--config",    f"{REPO_DIR}/conf/config.yaml",
        "--tablets",   f"{DATA_DIR}/metadata/tablets.json",
        "--output",    f"{DATA_DIR}/parallels/parallel_variants_auto.json",
        "--threshold", "1",
        "--verbose",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    print(f"WARNING: Cross-reference failed (exit {proc.returncode}) — continuing")

variants_path = Path(f"{DATA_DIR}/parallels/parallel_variants_auto.json")
if variants_path.exists():
    v = json.loads(variants_path.read_text())
    n_multi = sum(1 for p in v.get("passages", []) if p.get("n_tablets", 0) >= 2)
    print(f"\n  ✓ parallel_variants_auto.json: {n_multi} multi-tablet passages found")
else:
    print("  ✗ parallel_variants_auto.json not generated")

# ── Step 5: Passage report ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: Diachronic passage report")
print("=" * 60)
if variants_path.exists():
    proc = subprocess.Popen(
        [
            sys.executable, "-m", "hackingrongo.results.passage_report",
            "--input",        str(variants_path),
            "--output",       f"{REPO_DIR}/outputs/analysis/passage_reports",
            "--filter-score", "0.0",
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        print(f"WARNING: Passage report failed (exit {proc.returncode}) — continuing")
else:
    print("Skipped — parallel_variants_auto.json not found")

# ── Save to Drive ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Saving to Drive")
print("=" * 60)

for f in sorted(lm_dir.glob("*.json")):
    shutil.copy(f, Path(CHECKPOINTS_DIR) / f.name)
    print(f"  ✓ {f.name}")

if variants_path.exists():
    shutil.copy(variants_path, Path(CHECKPOINTS_DIR) / "parallel_variants_auto.json")
    print(f"  ✓ parallel_variants_auto.json")

passage_dir = Path(f"{REPO_DIR}/outputs/analysis/passage_reports")
if passage_dir.exists():
    n = 0
    for f in passage_dir.iterdir():
        if f.is_file() and f.suffix == ".html":
            shutil.copy(f, Path(CHECKPOINTS_DIR) / f.name)
            n += 1
    print(f"  ✓ passage_reports/ ({n} HTML files)")

print("\nZone C Cell 2 (MCMC decipherment) is ready to run.")

In [ ]:
# ── Zone B Cell 3: Reading order tests (with HTML report) ─────────────────────
# Four entropy tests for rongorongo transcription direction and recto/verso order.
# Tests 3 & 4 need only the corpus — they run immediately.
# Tests 1 & 2 need the sequence model from Zone A Cell 2 — skipped automatically if absent.
#
# Test 1: Conditional entropy asymmetry  H_forward vs H_reverse
# Test 2: N-gram model perplexity asymmetry (trained model, forward vs reversed)
# Test 3: Line-boundary entropy — are line breaks real compositional units?
# Test 4: Recto/verso ordering — resolves Pozdniakov's question from 1958
#
# Outputs written to Drive:
#   reading_order_results.json       ← structured test results (JSON)
#   reading_order_report.html        ← scholar-facing HTML report

import os, subprocess, sys, json, shutil
from pathlib import Path

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"

os.chdir(REPO_DIR)

RESULTS_JSON = Path(f"{REPO_DIR}/outputs/reading_order_results.json")
REPORT_HTML  = Path(f"{REPO_DIR}/outputs/analysis/reading_order_report.html")
CORPUS_DIR   = Path(f"{DATA_DIR}/corpus")
MODEL_PATH   = Path(f"{REPO_DIR}/outputs/zone_b/sequence_model.json")

if not CORPUS_DIR.exists():
    raise RuntimeError(f"Corpus directory not found: {CORPUS_DIR}\nRun Cell 3 first.")

# ── Tests 3 & 4 (no model required) ──────────────────────────────────────────
print("=" * 60)
print("READING ORDER TESTS 3 & 4 (corpus only)")
print("=" * 60)
subprocess.check_call(
    [
        sys.executable, "scripts/reading_order_tests.py",
        "--corpus", str(CORPUS_DIR),
        "--tests",  "3", "4",
        "--output", str(RESULTS_JSON),
    ],
    cwd=REPO_DIR,
)

# ── Tests 1 & 2 (sequence model required) ────────────────────────────────────
if MODEL_PATH.exists():
    print("\n" + "=" * 60)
    print("READING ORDER TESTS 1 & 2 (sequence model found)")
    print("=" * 60)
    subprocess.check_call(
        [
            sys.executable, "scripts/reading_order_tests.py",
            "--corpus", str(CORPUS_DIR),
            "--model",  str(MODEL_PATH),
            "--tests",  "1", "2", "3", "4",
            "--output", str(RESULTS_JSON),
        ],
        cwd=REPO_DIR,
    )
else:
    print(f"\n  Sequence model not found at {MODEL_PATH}")
    print("  Tests 1 & 2 skipped. Run Zone A Cell 1 to train the model, then re-run this cell.")

# ── Generate HTML report ──────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("GENERATING READING ORDER REPORT")
print("=" * 60)
subprocess.check_call(
    [
        sys.executable, "-m", "hackingrongo.results.reading_order_report",
        "--input",  str(RESULTS_JSON),
        "--output", str(REPORT_HTML),
    ],
    cwd=REPO_DIR,
)

# ── Display key results ───────────────────────────────────────────────────────
if RESULTS_JSON.exists():
    results = json.loads(RESULTS_JSON.read_text())
    t4 = results.get("test4", {})
    t3 = results.get("test3", {})
    if t4:
        print("\n" + "=" * 60)
        print("TEST 4  RECTO / VERSO ORDERING")
        print("=" * 60)
        print(f"  {t4.get('verdict_text', '(no verdict)')}")
        ppl_ab2 = t4.get("ppl_ab_bigram");  ppl_ba2 = t4.get("ppl_ba_bigram")
        ppl_ab3 = t4.get("ppl_ab_trigram"); ppl_ba3 = t4.get("ppl_ba_trigram")
        if all(v is not None for v in [ppl_ab2, ppl_ba2, ppl_ab3, ppl_ba3]):
            print(f"  Bigram  LOO-PPL   a→b : {ppl_ab2:.2f}   b→a : {ppl_ba2:.2f}")
            print(f"  Trigram LOO-PPL   a→b : {ppl_ab3:.2f}   b→a : {ppl_ba3:.2f}")
    if t3:
        print("\nTEST 3  LINE BOUNDARY ENTROPY")
        print(f"  {t3.get('verdict_text', '(no verdict)')}")

# ── Copy to Drive ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Copying to Drive")
print("=" * 60)
for src, name in [
    (RESULTS_JSON, "reading_order_results.json"),
    (REPORT_HTML,  "reading_order_report.html"),
]:
    if src.exists():
        shutil.copy(src, f"{CHECKPOINTS_DIR}/{name}")
        print(f"  ✓ {name}  ({src.stat().st_size / 1024:.0f} KB)")
    else:
        print(f"  — {name}  (not generated)")

In [ ]:
# ── Zone B Cell 4: Parallel passage ranking and download ──────────────────────
# Shows all parallel passages ranked by attestation count (number of tablets
# they appear on). The top passage is the best target for focused decipherment
# in Zone C Cell 2 — it has the most cross-tablet variant evidence to constrain MCMC.
#
# Outputs:
#   passage_reports/index.html   ← summary of all passages
#   passage_reports/{pid}.html   ← one file per passage (glyph images + alignment)
#
# Downloads index.html and the top passage directly to your browser.

import json, shutil
from pathlib import Path
from google.colab import files
from IPython.display import HTML, display

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"
PASSAGE_REPORTS = f"{REPO_DIR}/outputs/analysis/passage_reports"

variants_path = Path(f"{DATA_DIR}/parallels/parallel_variants_auto.json")

if not variants_path.exists():
    print("parallel_variants_auto.json not found — run Zone B Cell 2 first.")
else:
    raw = json.loads(variants_path.read_text())
    passages = raw.get("passages", raw if isinstance(raw, list) else [])

    # Rank by number of distinct tablets, then total variant count.
    ranked = sorted(
        [p for p in passages if isinstance(p, dict)],
        key=lambda p: (p.get("n_tablets", len(p.get("variants", []))), len(p.get("variants", []))),
        reverse=True,
    )

    print(f"{'ID':<10} {'Tablets':>7}  {'Variants':>8}  {'Length':>6}  Attested on")
    print("-" * 70)
    for p in ranked[:20]:
        pid      = p.get("passage_id", p.get("id", "?"))
        n_tab    = p.get("n_tablets", "?")
        n_var    = len(p.get("variants", []))
        canon    = p.get("canonical_form", [])
        n_glyphs = len(canon)
        tab_ids  = sorted({v.get("tablet_id", "?") for v in p.get("variants", [])})
        print(f"  {str(pid):<8} {str(n_tab):>7}  {n_var:>8}  {n_glyphs:>6}  {', '.join(tab_ids)}")

    if ranked:
        top = ranked[0]
        TOP_PASSAGE_ID = str(top.get("passage_id", top.get("id", "")))
        top_tabs = sorted({v.get("tablet_id", "?") for v in top.get("variants", [])})
        print(f"\n→ Top passage: {TOP_PASSAGE_ID}  ({top.get('n_tablets', '?')} tablets: {', '.join(top_tabs)})")
        print(f"  Set FOCUS_PASSAGE = \"{TOP_PASSAGE_ID}\" in Zone C Cell 2 to run focused decipherment.")
    else:
        TOP_PASSAGE_ID = ""
        print("No passages found.")

    # Download passage HTML reports if they were generated.
    report_dir = Path(PASSAGE_REPORTS)
    if report_dir.exists():
        html_files = sorted(report_dir.glob("*.html"))
        print(f"\n{len(html_files)} HTML report(s) in {PASSAGE_REPORTS}")

        # Save to Drive.
        drive_passage = Path(CHECKPOINTS_DIR) / "passage_reports"
        drive_passage.mkdir(parents=True, exist_ok=True)
        for f in html_files:
            shutil.copy(f, drive_passage / f.name)

        # Download index + top passage to browser.
        for fname in ["index.html"] + ([f"{TOP_PASSAGE_ID}.html"] if TOP_PASSAGE_ID else []):
            p = report_dir / fname
            if p.exists():
                files.download(str(p))
                print(f"  ⬇ {fname}")
            else:
                print(f"  — {fname} not found (passage report may not have been generated yet)")
    else:
        print(f"\nPassage reports directory not found: {PASSAGE_REPORTS}")
        print("Run Zone A Cell 2 Step 6 or Zone B Cell 2 Step 5 to generate them.")

In [ ]:
# ── Zone B Cell 5: Rebuild language models (rescue) ──────────────────────────
# Run this cell if Zone B Cell 2 fails at the LM step but corpus data is already
# present. Rebuilds language models without re-running the full pipeline.

import subprocess, sys, os
REPO_DIR = "/content/repo"
os.chdir(REPO_DIR)

proc = subprocess.Popen(
    [sys.executable, "scripts/build_language_models.py", "hydra.job.chdir=false"],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"Exit code: {proc.returncode}")

In [ ]:
# ── Zone C Cell 2: MCMC and beam search decipherment ──────────────────────────
# Requires: Zone B Cell 2 (language models built and saved to Drive).
# Runtime: ~10-20 min full corpus; ~3-5 min focused passage (T4 GPU).
#
# Set FOCUS_PASSAGE to the passage ID from Zone B Cell 4 output for targeted
# decipherment (faster convergence, tighter signal). Leave blank for full corpus.

import subprocess, sys, os, json, shutil
from pathlib import Path

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"

os.chdir(REPO_DIR)

# ── Set passage focus (copy from Cell 7c output, or leave blank) ──────────────
FOCUS_PASSAGE = ""   # e.g. "H-6" — leave blank to decode the full corpus

# ── Build command ─────────────────────────────────────────────────────────────
cmd = [
    sys.executable, "scripts/run_decipherment.py",
    "hydra.job.chdir=false",
    f"paths.corpus_dir={DATA_DIR}/corpus",
    f"paths.outputs_dir={REPO_DIR}/outputs",
    f"paths.lm_dir={DATA_DIR}/language_models",
    f"paths.parallel_variants={DATA_DIR}/parallels/parallel_variants_auto.json",
]
if FOCUS_PASSAGE.strip():
    cmd.append(f"--focus-passage={FOCUS_PASSAGE.strip()}")
    print(f"Focused decipherment on passage: {FOCUS_PASSAGE}")
else:
    print("Full-corpus decipherment ...")

proc = subprocess.Popen(
    cmd,
    cwd=REPO_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Decipherment failed (exit {proc.returncode})")

# ── Display top hypotheses ─────────────────────────────────────────────────────
ranking_path = Path(f"{REPO_DIR}/outputs/decipherment/ranking.json")
if ranking_path.exists():
    ranking = json.loads(ranking_path.read_text())
    top = ranking.get("top_hypotheses", ranking if isinstance(ranking, list) else [])
    print(f"\n── Top {min(5, len(top))} hypotheses ──────────────────────────────────────")
    for i, h in enumerate(top[:5], 1):
        score = h.get("normalised_score", h.get("score", 0))
        n_signs = len(h.get("phoneme_map", {}))
        print(f"  #{i}  score={score:.4f}  signs={n_signs}")
        sample = dict(list(h.get("phoneme_map", {}).items())[:6])
        print(f"       {sample}")

# ── Save to Drive ─────────────────────────────────────────────────────────────
decipher_dir = Path(f"{REPO_DIR}/outputs/decipherment")
if decipher_dir.exists():
    decipher_out = Path(CHECKPOINTS_DIR) / "decipherment"
    decipher_out.mkdir(parents=True, exist_ok=True)
    for fname in ["ranking.json", "ranking.csv", "ranking.md", "decipherment_report.html"]:
        src = decipher_dir / fname
        if src.exists():
            shutil.copy(src, decipher_out / fname)
            print(f"  ✓ Saved {fname} to Drive")
    hyp_files = sorted(decipher_dir.glob("hypothesis_*.json"))
    for f in hyp_files[:10]:
        shutil.copy(f, decipher_out / f.name)
    if hyp_files:
        print(f"  ✓ {len(hyp_files)} hypothesis files saved to Drive")

print("\nFinal Cell 1 (save + download) is ready to run.")

In [ ]:
# ── Cell 8c: HSP group structure analysis ────────────────────────────────
# Extracts sign substitution pairs from parallel passage variants,
# tests group closure, finds a minimal generator set, and assesses
# feasibility of Hidden Subgroup Problem framing for quantum decipherment.
#
# Requires: Zone B Cell 2 (parallel_variants_auto.json generated).
# Output saved to Drive: hsp_analysis.json

import subprocess, sys, os, json, shutil
from pathlib import Path

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"
os.chdir(REPO_DIR)

VARIANTS_PATH = Path(f"{DATA_DIR}/parallels/parallel_variants_auto.json")
HSP_OUT       = Path(f"{REPO_DIR}/outputs/hsp_analysis.json")

if not VARIANTS_PATH.exists():
    print("parallel_variants_auto.json not found — run Zone B Cell 2 first.")
else:
    proc = subprocess.Popen(
        [
            sys.executable, "scripts/hsp_analysis.py",
            "--parallels", str(VARIANTS_PATH),
            "--output",    str(HSP_OUT),
        ],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        print(f"WARNING: hsp_analysis failed (exit {proc.returncode})")

    if HSP_OUT.exists():
        result = json.loads(HSP_OUT.read_text())
        feas   = result.get("hsp_feasibility", {})
        subs   = result.get("substitutions", {})
        gens   = result.get("generators", [])
        clos   = result.get("closure", {})
        print(f"\n── HSP feasibility: {feas.get('level', '?').upper()} ──────────")
        print(f"  {feas.get('rationale', '')}")
        print(f"  Observed substitutions : {subs.get('n_pairs', '?')}")
        print(f"  Closure ratio          : {clos.get('closure_ratio', 0):.2f}")
        print(f"  Generator set size     : {len(gens)}")
        if gens:
            print(f"  Generators: {gens[:8]}")
        shutil.copy(HSP_OUT, Path(CHECKPOINTS_DIR) / "hsp_analysis.json")
        print(f"  ✓ hsp_analysis.json saved to Drive")

In [ ]:
# ── Zone C Cell 4: Tablet D reconstruction ──────────────────────────────────
# Runs the Échancrée (Tablet D) uncertain-sign reconstruction pipeline.
# Requires: Zone C Cell 2 (ranking.json) and Zone A Cell 2 (sequence_model.json).
# Gracefully degrades if either is missing — always writes JSON + HTML.
#
# Outputs saved to Drive:
#   reconstruction/tablet_d_reconstruction.json
#   reconstruction/tablet_d_reconstruction_report.html

import subprocess, sys, os, shutil, json
from pathlib import Path

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
os.chdir(REPO_DIR)

proc = subprocess.Popen(
    [sys.executable, "scripts/reconstruct_tablet_d.py",
     "--corpus-dir", f"{REPO_DIR}/data/corpus",
     "--model",      f"{REPO_DIR}/outputs/zone_b/sequence_model.json",
     "--ranking",    f"{REPO_DIR}/outputs/decipherment/ranking.json",
     "--glyphs-dir", f"{REPO_DIR}/data/glyphs",
     "--output",     f"{REPO_DIR}/outputs/reconstruction/"],
    cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()

result_path = Path(f"{REPO_DIR}/outputs/reconstruction/tablet_d_reconstruction.json")
if result_path.exists():
    result = json.loads(result_path.read_text())
    summary = result.get("summary", {})
    print(f"\n── Tablet D Reconstruction Summary ─────────────────────────")
    print(f"  Targets identified    : {summary.get('n_targets', 0)}")
    print(f"  Convergent candidates : {summary.get('n_convergent', 0)}")
    print(f"  Top candidate         : {summary.get('top_candidate', '\u2014')}")
    candidates = result.get("convergent_candidates", [])
    for c in candidates:
        print(f"  {c['barthel_raw']:<10} score={c['convergence_score']:.2f}  "
              f"mcmc={c.get('mcmc_phoneme','\u2014')}  "
              f"seq_top1={c['sequence_top_k'][0]['sign'] if c['sequence_top_k'] else '\u2014'}")

recon_ckpt = Path(CHECKPOINTS_DIR) / "reconstruction"
recon_ckpt.mkdir(exist_ok=True)
for fname in ["tablet_d_reconstruction.json", "tablet_d_reconstruction_report.html"]:
    src = Path(f"{REPO_DIR}/outputs/reconstruction/{fname}")
    if src.exists():
        shutil.copy(src, recon_ckpt / fname)
        print(f"  \u2713 Saved {fname} to Drive")

In [ ]:
# ── Cell 9: Save all outputs to Drive + download key files ──────────────────
# Run as the final cell after the pipeline completes.

import shutil, json, os
from pathlib import Path
from google.colab import files

REPO_DIR        = "/content/repo"
CHECKPOINTS_DIR = "/content/drive/MyDrive/hackingrongo_checkpoints"
DATA_DIR        = f"{REPO_DIR}/data"
OUTPUT_DIR      = Path(CHECKPOINTS_DIR)  # Drive is the persistent output store
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("── Saving outputs to Drive ──────────────────────────────────")

def _save(src: Path, label: str) -> bool:
    if src.exists():
        dest = OUTPUT_DIR / src.name
        shutil.copy(src, dest)
        print(f"  ✓ {label:45s} {src.stat().st_size/1024:.0f} KB")
        return True
    print(f"  — {label:45s} not generated")
    return False

# Checkpoints
for f in sorted(Path(f"{REPO_DIR}/outputs/checkpoints").glob("*.pt")):
    _save(f, f.name)

# Zone A
for fname in ["umap_embeddings.png", "cluster_vs_barthel.csv", "cluster_vs_barthel.json"]:
    _save(Path(f"{REPO_DIR}/outputs/analysis/{fname}"), fname)
_save(Path(f"{REPO_DIR}/outputs/analysis/divergence_report.html"), "divergence_report.html")
_save(Path(f"{REPO_DIR}/outputs/analysis/stratum_glyph_report.html"), "stratum_glyph_report.html")

# Zone B: compound detection
for fname in ["compound_candidates.json", "compound_validation.json", "compound_report.html"]:
    _save(Path(f"{REPO_DIR}/outputs/analysis/{fname}"), fname)

# Zone B: parallel passages
_save(Path(f"{DATA_DIR}/parallels/parallel_variants_auto.json"), "parallel_variants_auto.json")

# Zone B: passage reports
passage_dir = Path(f"{REPO_DIR}/outputs/analysis/passage_reports")
if passage_dir.exists():
    passage_out = OUTPUT_DIR / "passage_reports"
    passage_out.mkdir(exist_ok=True)
    n_passage = 0
    for f in sorted(passage_dir.glob("*.html")):
        shutil.copy(f, passage_out / f.name)
        n_passage += 1
    print(f"  ✓ {'passage_reports/':45s} {n_passage} HTML files")
else:
    print(f"  — {'passage_reports/':45s} not generated")

# Zone B: astronomical
_save(Path(f"{REPO_DIR}/outputs/zone_b/astronomical_candidates.json"), "astronomical_candidates.json")
_save(Path(f"{REPO_DIR}/outputs/analysis/astronomical_report.html"), "astronomical_report.html")

# Quantum
_save(Path(f"{REPO_DIR}/outputs/zone_b/pgood_analysis.json"), "pgood_analysis.json")

# Zone C
decipher_dir = Path(f"{REPO_DIR}/outputs/decipherment")
decipher_out = OUTPUT_DIR / "decipherment"
decipher_out.mkdir(exist_ok=True)
for fname in ["ranking.json", "ranking.csv", "ranking.md", "decipherment_report.html"]:
    src = decipher_dir / fname
    if src.exists():
        shutil.copy(src, decipher_out / fname)
        print(f"  ✓ {fname:45s} {src.stat().st_size/1024:.0f} KB")
    else:
        print(f"  — {fname:45s} not generated")

if decipher_dir.exists():
    hyp_files = sorted(decipher_dir.glob("hypothesis_*.json"))
    for f in hyp_files[:10]:
        shutil.copy(f, decipher_out / f.name)
    if hyp_files:
        print(f"  ✓ {'hypothesis_*.json (top 10)':45s} {len(hyp_files)} total")

# Language models
lm_dir = Path(f"{DATA_DIR}/language_models")
if lm_dir.exists():
    lm_out = OUTPUT_DIR / "language_models"
    lm_out.mkdir(exist_ok=True)
    lm_files = sorted(lm_dir.glob("*.json"))
    for f in lm_files:
        shutil.copy(f, lm_out / f.name)
    total_kb = sum(f.stat().st_size for f in lm_files) / 1024
    print(f"  ✓ {'language_models/':45s} {len(lm_files)} files, {total_kb:.0f} KB")

# ── Summary ───────────────────────────────────────────────────────────────────
all_files = [f for f in OUTPUT_DIR.rglob("*") if f.is_file()]
total_mb  = sum(f.stat().st_size for f in all_files) / 1024 / 1024
html_files = [f for f in all_files if f.suffix == ".html"]

print(f"\n── Summary ──────────────────────────────────────────────────")
print(f"  Total files:  {len(all_files)}")
print(f"  Total size:   {total_mb:.1f} MB")
print(f"  Drive path:   {CHECKPOINTS_DIR}")
print(f"  HTML reports: {len(html_files)}")
for f in sorted(html_files):
    print(f"    {f.name:45s} {f.stat().st_size/1024:.0f} KB")

# ── Download to local machine ─────────────────────────────────────────────────
print(f"\n── Downloading key files to local machine ───────────────────")
print("(Files will appear in your browser's download folder)")

DOWNLOAD_THESE = [
    OUTPUT_DIR / "divergence_report.html",
    OUTPUT_DIR / "compound_report.html",
    OUTPUT_DIR / "astronomical_report.html",
    OUTPUT_DIR / "stratum_glyph_report.html",
    OUTPUT_DIR / "passage_reports" / "index.html",
    OUTPUT_DIR / "decipherment" / "decipherment_report.html",
    OUTPUT_DIR / "decipherment" / "ranking.json",
    OUTPUT_DIR / "pgood_analysis.json",
]
for f in DOWNLOAD_THESE:
    if f.exists():
        files.download(str(f))
        print(f"  ⬇ {f.name}")
    else:
        print(f"  — {f.name} (not generated — skip)")
